# InfinityDiff Baseline for Navier–Stokes Next-Step Forecasting

This notebook provides a self-contained baseline that adapts the [$\infty$-Diff](https://arxiv.org/abs/2303.18242) architecture to the sparse Navier–Stokes forecasting setup. The goal is to predict the full vorticity field at time $t+1$ from a sparsely observed field at time $t$ under a fixed per-sample sparsity mask.


## 1. Environment setup

The notebook relies on the dependencies that already ship with the InfinityDiff repository. In particular, it assumes that the environment satisfies `requirements.yml` and that the dataset has been preprocessed into the ragged mmap format described by the Navier–Stokes benchmark.

> **Note:** The notebook is designed to be run from the root of the repository so that relative imports (e.g. `from models import SparseUNet`) resolve correctly.


In [ ]:
import os
import math
import json
from dataclasses import dataclass
from typing import Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from mmap_ninja import RaggedMmap

from models import SparseUNet


## 2. Configuration

Adjust the paths and hyperparameters below to match your experimental setting. The defaults mirror the PerceiverIO baseline so that results are comparable.


In [ ]:
@dataclass
class Config:
    # Data parameters
    base_dir: str = "/pscratch/sd/d/dpark1/NSData"  # change to your storage location
    split_name: str = "100k"  # folder name inside the base directory
    sparsity: float = 0.20  # e.g. 0.20 -> 20% visible per sample
    mean: float = 0.0
    std: float = 2.4036
    normalize: bool = True
    to_unit_range: bool = False
    fixed_mask_per_sample: bool = True
    mask_seed: int = 0
    train_pairs: int = 4900
    val_pairs: int = 4900
    batch_size: int = 32
    num_workers: int = 4

    # Grid parameters
    height: int = 64
    width: int = 64
    channels: int = 1

    # Model parameters (InfinityDiff core)
    uno_resolution: int = 64
    model_nf: int = 64
    model_time_dim: int = 256
    num_conv_blocks: int = 3
    knn_neighbours: int = 3
    uno_mults: Tuple[int, ...] = (1, 2, 4, 8)
    uno_blocks_per_level: Tuple[int, ...] = (2, 2, 2, 2)
    uno_attn_resolutions: Tuple[int, ...] = (16, 8)
    uno_dropout_from_resolution: int = 16
    uno_dropout: float = 0.0
    kernel_size: int = 5
    backend: str = "torch_dense"
    optimise_dense: bool = True

    # Optimisation
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    epochs: int = 50
    learning_rate: float = 2e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0

    # Checkpointing
    checkpoint_dir: str = "NavierStokesCheckpoints/InfinityDiff"
    run_tag: str = "10pct_20pct"

    # Training switches
    amp: bool = True
    log_every: int = 50
    eval_every: int = 1

CONFIG = Config()
CONFIG


## 3. Dataset utilities

The dataset loader mirrors the PerceiverIO baseline: we work with a ragged mmap of Navier–Stokes trajectories, build fixed train/validation splits, and apply a deterministic sparsity mask per sample. The dataset returns the sparse observation, the dense target, the boolean mask, and a unique sample identifier.


In [ ]:
def exists(x):
    return x is not None


def default(val, d):
    return val if exists(val) else d


def build_index_pairs(mem, max_pairs, seed):
    rng = np.random.default_rng(seed)
    num_items_total = len(mem)
    assert num_items_total > 0
    T = int(mem[0].shape[0])
    assert T >= 2
    pairs_per_item = T - 1
    max_items = math.ceil(max_pairs / pairs_per_item)
    chosen_items = np.sort(rng.choice(num_items_total, size=max_items, replace=False))
    pairs = [(int(it), int(t)) for it in chosen_items for t in range(T - 1)]
    pairs = np.asarray(pairs[:max_pairs], dtype=np.int64)
    print(f"[pairs] selected {len(chosen_items)} items -> {len(pairs)} pairs")
    return pairs, T


class NavierStokesSparseNextStep(Dataset):
    def __init__(
        self,
        memmap: RaggedMmap,
        index_pairs: np.ndarray,
        *,
        sparsity: float,
        normalize: bool,
        mean: float,
        std: float,
        to_unit_range: bool,
        seed: int,
        fixed_mask_per_sample: bool,
        mask_seed: int,
        verbose: bool = True,
    ):
        self.mem = memmap
        self.index_pairs = np.asarray(index_pairs, dtype=np.int64)
        self.normalize = bool(normalize)
        self.mean = float(mean)
        self.std = float(std)
        self.to_unit_range = bool(to_unit_range)
        self.sparsity = float(sparsity)
        self.global_rng = np.random.default_rng(seed)
        self.fixed_mask_per_sample = bool(fixed_mask_per_sample)
        self.mask_seed = int(mask_seed)

        self.T = int(memmap[int(self.index_pairs[0, 0])].shape[0])
        if verbose:
            print(
                f"[dataset] pairs={len(self.index_pairs)}, T={self.T}, "
                f"sparsity={self.sparsity}, normalize={self.normalize}, "
                f"unit_range={self.to_unit_range}, fixed_mask={self.fixed_mask_per_sample}"
            )

    def __len__(self) -> int:
        return len(self.index_pairs)

    def _normalise(self, arr: np.ndarray) -> np.ndarray:
        arr = (arr - self.mean) / self.std if self.normalize else arr
        if self.to_unit_range:
            arr = np.tanh(arr)
        return arr

    def __getitem__(self, idx: int):
        item_idx, t = map(int, self.index_pairs[idx])
        seq = self.mem[item_idx]
        x = np.asarray(seq[t], dtype=np.float32)
        y = np.asarray(seq[t + 1], dtype=np.float32)

        x = self._normalise(x)
        y = self._normalise(y)

        sample_id = int(item_idx * self.T + t)
        if self.fixed_mask_per_sample:
            rng = np.random.default_rng(self.mask_seed + sample_id)
        else:
            rng = self.global_rng
        mask = rng.random(x.shape, dtype=np.float32) < self.sparsity

        x_sparse = x * mask.astype(np.float32)

        x_sparse = torch.from_numpy(x_sparse[None, ...])  # (1, H, W)
        y = torch.from_numpy(y[None, ...])
        mask = torch.from_numpy(mask.astype(np.bool_))
        sid = torch.tensor(sample_id, dtype=torch.long)
        return x_sparse, y, mask, sid


### 3.1 Fixed train/validation splits

We follow the deterministic splitting scheme used in other baselines so that the train and validation sets are disjoint and reproducible.


In [ ]:
def build_fixed_item_splits(mem, target_train_pairs, val_pairs, train_seed=0, val_seed=12345):
    num_items_total = len(mem)
    assert num_items_total > 0
    T = int(mem[0].shape[0])
    assert T >= 2
    pairs_per_item = T - 1

    items_for_val = int(math.ceil(val_pairs / pairs_per_item))
    items_for_train = int(math.ceil(target_train_pairs / pairs_per_item))

    rng_val = np.random.default_rng(val_seed)
    val_items = np.sort(rng_val.choice(num_items_total, size=items_for_val, replace=False))

    remaining = np.setdiff1d(np.arange(num_items_total), val_items, assume_unique=True)
    assert len(remaining) >= items_for_train

    rng_train = np.random.default_rng(train_seed)
    train_items = np.sort(rng_train.choice(remaining, size=items_for_train, replace=False))

    def pairs_from_items(items):
        return np.asarray([(int(it), int(t)) for it in items for t in range(T - 1)], dtype=np.int64)

    train_pairs = pairs_from_items(train_items)[:target_train_pairs]
    val_pairs = pairs_from_items(val_items)[:val_pairs]

    assert set(map(tuple, train_pairs)).isdisjoint(set(map(tuple, val_pairs)))

    print(
        f"[splits] T={T} -> pairs/item={pairs_per_item} | "
        f"train_items={len(train_items)} -> train_pairs={len(train_pairs)} | "
        f"val_items={len(val_items)} -> val_pairs={len(val_pairs)}"
    )

    return train_pairs, val_pairs, T, train_items, val_items


## 4. Data module

Load the ragged mmap, build the fixed splits, and prepare PyTorch data loaders. The sparse observation and mask are concatenated into two input channels for the InfinityDiff model.


In [ ]:
memmap_path = os.path.join(CONFIG.base_dir, CONFIG.split_name)
mem = RaggedMmap(memmap_path, mode="r")
train_pairs, val_pairs, T, train_items, val_items = build_fixed_item_splits(
    mem,
    target_train_pairs=CONFIG.train_pairs,
    val_pairs=CONFIG.val_pairs,
)

train_ds = NavierStokesSparseNextStep(
    mem,
    train_pairs,
    sparsity=CONFIG.sparsity,
    normalize=CONFIG.normalize,
    mean=CONFIG.mean,
    std=CONFIG.std,
    to_unit_range=CONFIG.to_unit_range,
    seed=42,
    fixed_mask_per_sample=CONFIG.fixed_mask_per_sample,
    mask_seed=CONFIG.mask_seed,
    verbose=True,
)

val_ds = NavierStokesSparseNextStep(
    mem,
    val_pairs,
    sparsity=CONFIG.sparsity,
    normalize=CONFIG.normalize,
    mean=CONFIG.mean,
    std=CONFIG.std,
    to_unit_range=CONFIG.to_unit_range,
    seed=43,
    fixed_mask_per_sample=CONFIG.fixed_mask_per_sample,
    mask_seed=CONFIG.mask_seed,
    verbose=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG.batch_size,
    shuffle=True,
    num_workers=CONFIG.num_workers,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=CONFIG.num_workers,
    pin_memory=True,
)

CONFIG.device


## 5. InfinityDiff forecasting model

We instantiate `SparseUNet` from the repository and operate it in dense mode. The model receives a two-channel tensor consisting of the masked observation and the binary mask, and predicts the full field at the next time step. A zero diffusion time embedding is used because we treat the network deterministically.


In [ ]:
model = SparseUNet(
    channels=2,  # observation + mask
    nf=CONFIG.model_nf,
    time_emb_dim=CONFIG.model_time_dim,
    img_size=CONFIG.height,
    num_conv_blocks=CONFIG.num_conv_blocks,
    knn_neighbours=CONFIG.knn_neighbours,
    uno_res=CONFIG.uno_resolution,
    uno_mults=CONFIG.uno_mults,
    z_dim=None,
    out_channels=CONFIG.channels,
    conv_type="conv",
    depthwise_sparse=True,
    kernel_size=CONFIG.kernel_size,
    backend=CONFIG.backend,
    blocks_per_level=CONFIG.uno_blocks_per_level,
    attn_res=CONFIG.uno_attn_resolutions,
    dropout_res=CONFIG.uno_dropout_from_resolution,
    dropout=CONFIG.uno_dropout,
    optimise_dense=CONFIG.optimise_dense,
)
model = model.to(CONFIG.device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG.learning_rate,
    weight_decay=CONFIG.weight_decay,
)

scaler = torch.cuda.amp.GradScaler(enabled=CONFIG.amp)
criterion = nn.MSELoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


## 6. Metrics

We track MSE, RMSE, MAE, and CRPS (with a deterministic ensemble) during validation. The unnormalised versions are useful for comparing to raw-field baselines.


In [ ]:
@torch.no_grad()
def compute_metrics(pred: torch.Tensor, target: torch.Tensor, std: float) -> dict:
    mse = F.mse_loss(pred, target, reduction="mean")
    mae = F.l1_loss(pred, target, reduction="mean")
    rmse = mse.sqrt()
    preds_K = pred.unsqueeze(0).repeat(10, 1, 1, 1, 1)
    term1 = (preds_K - target.unsqueeze(0)).abs().mean(dim=0)
    diffs = (preds_K.unsqueeze(0) - preds_K.unsqueeze(1)).abs()
    term2 = 0.5 * diffs.mean(dim=(0, 1))
    crps = (term1 - term2).mean()
    return {
        "mse": mse.item(),
        "rmse": rmse.item(),
        "mae": mae.item(),
        "crps": crps.item(),
        "rmse_unnorm": rmse.item() * std,
        "mae_unnorm": mae.item() * std,
        "crps_unnorm": crps.item() * std,
    }


## 7. Training and validation loops

Set `RUN_TRAINING = True` to launch optimisation. Progress is logged every `CONFIG.log_every` steps, and the best checkpoint is tracked using the validation MSE.


In [ ]:
RUN_TRAINING = False  # flip to True to train the model

best_val_mse = float("inf")
best_epoch = -1
os.makedirs(os.path.join(CONFIG.checkpoint_dir, CONFIG.run_tag), exist_ok=True)
checkpoint_path = os.path.join(CONFIG.checkpoint_dir, CONFIG.run_tag, "best.pt")

if RUN_TRAINING:
    for epoch in range(1, CONFIG.epochs + 1):
        model.train()
        epoch_loss = 0.0
        progress = tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG.epochs} [train]")
        for step, (x_sparse, y, mask, sid) in enumerate(progress, start=1):
            x_sparse = x_sparse.to(CONFIG.device, non_blocking=True)
            y = y.to(CONFIG.device, non_blocking=True)
            mask = mask.to(CONFIG.device, non_blocking=True)

            inp = torch.cat([x_sparse, mask.unsqueeze(1).float()], dim=1)
            t = torch.zeros(inp.size(0), device=CONFIG.device)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=CONFIG.amp):
                pred = model(inp, t)
                loss = criterion(pred, y)

            scaler.scale(loss).backward()
            if CONFIG.grad_clip > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG.grad_clip)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            if step % CONFIG.log_every == 0:
                progress.set_postfix(loss=loss.item())
        progress.close()

        if epoch % CONFIG.eval_every == 0:
            model.eval()
            val_losses = []
            metrics_accum = []
            with torch.no_grad():
                for x_sparse, y, mask, sid in tqdm(val_loader, desc=f"Epoch {epoch} [val]"):
                    x_sparse = x_sparse.to(CONFIG.device, non_blocking=True)
                    y = y.to(CONFIG.device, non_blocking=True)
                    mask = mask.to(CONFIG.device, non_blocking=True)

                    inp = torch.cat([x_sparse, mask.unsqueeze(1).float()], dim=1)
                    t = torch.zeros(inp.size(0), device=CONFIG.device)

                    with torch.cuda.amp.autocast(enabled=CONFIG.amp):
                        pred = model(inp, t)
                        loss = criterion(pred, y)
                    val_losses.append(loss.item())
                    metrics_accum.append(compute_metrics(pred, y, CONFIG.std))

            mean_loss = float(np.mean(val_losses))
            mean_metrics = {k: float(np.mean([m[k] for m in metrics_accum])) for k in metrics_accum[0]}

            print(
                f"[Epoch {epoch}] train_loss={epoch_loss / len(train_loader):.4f} | "
                f"val_mse={mean_metrics['mse']:.4f} | val_rmse={mean_metrics['rmse']:.4f} | "
                f"val_mae={mean_metrics['mae']:.4f} | val_crps={mean_metrics['crps']:.4f}"
            )

            if mean_metrics['mse'] < best_val_mse:
                best_val_mse = mean_metrics['mse']
                best_epoch = epoch
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "config": CONFIG.__dict__,
                        "metrics": mean_metrics,
                    },
                    checkpoint_path,
                )
                print(f"  -> New best checkpoint saved to {checkpoint_path}")

    print(f"Training complete. Best val MSE={best_val_mse:.4f} at epoch {best_epoch}")
else:
    print("Training skipped. Set RUN_TRAINING = True to start optimisation.")


## 8. Evaluation helper

Reload the best checkpoint (if present) and run the validation metrics once more. This block can be executed independently after training.


In [ ]:
@torch.no_grad()
def evaluate_checkpoint(checkpoint_path: str):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    payload = torch.load(checkpoint_path, map_location=CONFIG.device)
    model.load_state_dict(payload["model_state_dict"])
    model.eval()

    val_metrics = []
    for x_sparse, y, mask, sid in tqdm(val_loader, desc="Evaluation"):
        x_sparse = x_sparse.to(CONFIG.device, non_blocking=True)
        y = y.to(CONFIG.device, non_blocking=True)
        mask = mask.to(CONFIG.device, non_blocking=True)

        inp = torch.cat([x_sparse, mask.unsqueeze(1).float()], dim=1)
        t = torch.zeros(inp.size(0), device=CONFIG.device)

        with torch.cuda.amp.autocast(enabled=CONFIG.amp):
            pred = model(inp, t)
        val_metrics.append(compute_metrics(pred, y, CONFIG.std))

    mean_metrics = {k: float(np.mean([m[k] for m in val_metrics])) for k in val_metrics[0]}
    print(json.dumps(mean_metrics, indent=2))
    return mean_metrics

# Example usage:
# metrics = evaluate_checkpoint(checkpoint_path)


## 9. Visualisation

Plot the sparse input, the ground-truth target, and the model prediction for a validation example. This cell assumes that the model weights have already been loaded (either from training in-session or via `evaluate_checkpoint`).


In [ ]:
@torch.no_grad()
def visualize_example(loader, num_samples: int = 1, seed: Optional[int] = None):
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(loader.dataset), size=num_samples, replace=False)

    rows = []
    for idx in indices:
        x_sparse, y, mask, sid = loader.dataset[idx]
        x_sparse = x_sparse.unsqueeze(0).to(CONFIG.device)
        y = y.unsqueeze(0).to(CONFIG.device)
        mask = mask.unsqueeze(0).to(CONFIG.device)

        inp = torch.cat([x_sparse, mask.unsqueeze(1).float()], dim=1)
        t = torch.zeros(inp.size(0), device=CONFIG.device)

        with torch.cuda.amp.autocast(enabled=CONFIG.amp):
            pred = model(inp, t)

        pred_img = pred[0, 0].detach().cpu().numpy()
        sparse_img = x_sparse[0, 0].detach().cpu().numpy()
        mask_img = mask[0, 0].detach().cpu().numpy()
        target_img = y[0, 0].detach().cpu().numpy()

        rows.append((sparse_img, mask_img, target_img, pred_img))

    fig, axes = plt.subplots(num_samples, 4, figsize=(12, 3 * num_samples))
    if num_samples == 1:
        axes = axes[None, :]
    for row_idx, (sparse_img, mask_img, target_img, pred_img) in enumerate(rows):
        ax = axes[row_idx]
        ax[0].imshow(sparse_img, cmap="viridis")
        ax[0].set_title("x_t (sparse)")
        ax[1].imshow(mask_img, cmap="gray")
        ax[1].set_title("mask")
        ax[2].imshow(target_img, cmap="viridis")
        ax[2].set_title("y_{t+1} (target)")
        ax[3].imshow(pred_img, cmap="viridis")
        ax[3].set_title("prediction")
        for a in ax:
            a.axis("off")
    plt.tight_layout()
    plt.show()

# Example usage:
# visualize_example(val_loader, num_samples=1, seed=0)
